In [0]:
# Restart Python to reload modules
dbutils.library.restartPython()

In [0]:
# ============================================================
# 04_streaming_incremental.py
# Phase 2 — Structured Streaming + Checkpointing
# ============================================================

import sys
sys.path.append("/Workspace/Users/jaswanth.vudduru@gmail.com/taxi-medallion-pipeline")
from config import STREAM_SOURCE, STREAM_BRONZE, CHECKPOINT_PATH

# -----------------------------------------------------------
# STEP 1: Drop batch 1 into the stream source folder
# -----------------------------------------------------------
dbutils.fs.put(f"{STREAM_SOURCE}batch1.json",
"""{"vendor_id":"1","pickup_datetime":"2024-01-01 10:00:00","passenger_count":2,"trip_distance":5.2,"fare_amount":18.5,"payment_type":"1"}
{"vendor_id":"2","pickup_datetime":"2024-01-01 10:15:00","passenger_count":1,"trip_distance":3.1,"fare_amount":12.0,"payment_type":"2"}""",
overwrite=True)

print("✅ Batch 1 dropped into stream source")


In [0]:

# -----------------------------------------------------------
# STEP 2: Define the stream — reads NEW files as they arrive
# -----------------------------------------------------------
from utils.schemas import bronze_schema

stream_df = (spark.readStream
             .schema(bronze_schema)
             .option("mode", "PERMISSIVE")
             .option("columnNameOfCorruptRecord", "_corrupt_record")
             .json(STREAM_SOURCE))


In [0]:

# -----------------------------------------------------------
# STEP 3: Write stream to Delta with checkpoint
# -----------------------------------------------------------
# trigger(availableNow=True) = process all waiting files then stop
# This is the right mode for Community Edition
query = (stream_df
         .filter(stream_df._corrupt_record.isNull())
         .drop("_corrupt_record")
         .writeStream
         .format("delta")
         .outputMode("append")
         .option("checkpointLocation", CHECKPOINT_PATH)
         .trigger(availableNow=True)
         .start(STREAM_BRONZE))

query.awaitTermination()
print(f"✅ Stream batch 1 processed. Rows: {spark.read.format('delta').load(STREAM_BRONZE).count()}")


In [0]:

# -----------------------------------------------------------
# STEP 4: Show checkpoint files (this is what enables fault tolerance)
# -----------------------------------------------------------
print("\n--- CHECKPOINT CONTENTS ---")
display(dbutils.fs.ls(CHECKPOINT_PATH))
# You'll see: commits/, offsets/, metadata
# These track exactly which files were processed and at what offset


In [0]:

# -----------------------------------------------------------
# STEP 5: Drop batch 2 — stream picks up ONLY new file
# -----------------------------------------------------------
dbutils.fs.put(f"{STREAM_SOURCE}batch2.json",
"""{"vendor_id":"1","pickup_datetime":"2024-01-02 09:00:00","passenger_count":4,"trip_distance":8.1,"fare_amount":30.0,"payment_type":"1"}
{"vendor_id":"2","pickup_datetime":"2024-01-02 09:30:00","passenger_count":2,"trip_distance":6.5,"fare_amount":22.5,"payment_type":"2"}""",
overwrite=True)

# Re-run the stream — it reads checkpoint, skips batch1.json, processes only batch2.json
query2 = (stream_df
          .filter(stream_df._corrupt_record.isNull())
          .drop("_corrupt_record")
          .writeStream
          .format("delta")
          .outputMode("append")
          .option("checkpointLocation", CHECKPOINT_PATH)  # SAME checkpoint path
          .trigger(availableNow=True)
          .start(STREAM_BRONZE))

query2.awaitTermination()
print(f"✅ Stream batch 2 processed. Total rows now: {spark.read.format('delta').load(STREAM_BRONZE).count()}")
# Should be 4 total — 2 from batch1 + 2 from batch2 (no duplicates)


In [0]:

# -----------------------------------------------------------
# STEP 6: Simulate crash recovery — explain what happens
# -----------------------------------------------------------
print("""
FAULT TOLERANCE EXPLANATION:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
If the stream crashes mid-way through batch2.json:

1. Checkpoint /offsets/ records: "I started processing batch2.json at offset 0"
2. Checkpoint /commits/ records: "I have NOT committed batch2.json yet"
3. On restart → Spark sees uncommitted offset → reprocesses batch2.json
4. Delta's ACID guarantee prevents duplicate writes (idempotent write)

Without checkpointLocation:
→ Restart = start from scratch = reprocess ALL files = duplicate data
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")